# CREATE CATALOG AND SCHEMA 

In [0]:
from pyspark.sql.functions import *


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS HEALTHCARE;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE.SILVER;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE.GOLD;

# READ BRONZE LAYER DATA - ENCOUNTERS

In [0]:
df_encounters=spark.read.option("multiline", "true").json("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/encounters.json")

In [0]:
df_encounters.limit(5).display()

In [0]:
df_bridge= df_encounters.select(
    col("encounter_id"),
    col("mrn"),
    explode(col("diagnoses")).alias("diag")
)
df_bridge.limit(5).display()

# Generate the bridge dataframe of encounter and diagnosis(one patient/encounter can have multiple diagnosis)

In [0]:
df_bridge=df_bridge.select (
    col("encounter_id"),
    col("mrn"),
    col("diag.icd10_code").alias("diag_code"),
    col("diag.diagnosis_type"),
    col("diag.diagnosis_rank")
)
df_bridge.limit(5).display()

## Write the encounter diagnosis bridge table to silver layer

In [0]:
df_bridge.write.format("parquet")\
    .mode("overwrite")\
        .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/enc_diag_bridge")

# Implement Hashing for Patient ID- Data Masking

Steps:
1. Connect to databricks CLI using workspace ID and access token
2. Create the scope for the secret(Eg:phi-scope)
3. Create the secret in the scope :
## databricks secrets put --scope phi-scope --key hash_salt --string-value "$(openssl rand -hex 32)"

# READ THE BRONZE DATA - PATIENT

In [0]:
df_mpi_patient=spark.read.format("parquet")\
.option("inferSchema", "true")\
    .load("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/raw_mpi_patient.parquet")
df_mpi_patient.limit(5).display()

In [0]:
salt = dbutils.secrets.get(scope="phi-scope", key="hash_salt")
print(salt)

In [0]:
df_mpi_patient = df_mpi_patient.withColumn(
    "patient_id_hash",
    sha2(concat(col("mrn"), lit(salt)), 256)
)

In [0]:
df_mpi_patient_conformed=df_mpi_patient.select("mrn","patient_id_hash","first_name","last_name","date_of_birth","gender","race","ethnicity","address_line1","city","state","zip_code","ssn","phone_number","email")
df_mpi_patient_conformed.limit(5).display()

## Data cleansing

In [0]:
df_mpi_patient_conformed=df_mpi_patient_conformed.filter(col("date_of_birth")<=current_date())

In [0]:
df_mpi_patient_conformed=df_mpi_patient_conformed.withColumn("date_of_birth",col("date_of_birth").cast("date"))

In [0]:
df_mpi_patient_conformed.limit(5).display()

## Write MPI patient to silver layer

In [0]:
df_mpi_patient_conformed.write.format("parquet")\
    .mode("overwrite")\
        .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/patient")

In [0]:
df_encounters.printSchema()

## TypeCasting- Encounters

In [0]:
from pyspark.sql.types import * 
df_encounters=df_encounters.withColumn("admit_date",col("admit_date").cast("date"))\
.withColumn("discharge_date",col("discharge_date").cast("date"))\
    .withColumn("rendering_npi",col("rendering_npi").cast(StringType()))\
        .withColumn("total_charge_amount",col("total_charge_amount").cast(DecimalType(12, 2)))\
            .withColumn("_ingest_timestamp",col("_ingest_timestamp").cast("timestamp"))

In [0]:
df_encounters.limit(5).display()

## Write encounters into silver layer

In [0]:
df_encounters.write.format("parquet")\
    .mode("overwrite")\
    .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/encounters")

# READ THE BRONZE DATA - LAB RESULT

In [0]:
df_lab_result = spark.read.format("parquet")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/raw_lab_result.parquet")

In [0]:
df_lab_result.limit(5).display()

In [0]:
df_lab_result=df_lab_result.withColumn("result_date",col("result_date").cast("date"))\
    .withColumn("_ingest_timestamp",col("_ingest_timestamp").cast("timestamp"))

In [0]:
df_lab_result.limit(5).display()

In [0]:
df_lab_result.count()

## Write the Lab Result into silver layer

In [0]:
df_lab_result.write.format("parquet")\
    .mode("overwrite")\
    .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/lab_result")

# READ THE BRONZE LAYER - PHARMACY MEDICATION ORDER

In [0]:
df_pharmacy_medication=spark.read.format("parquet")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/raw_pharmacy_medication_order.parquet")

In [0]:
df_pharmacy_medication.limit(5).display()

In [0]:
df_pharmacy_medication=df_pharmacy_medication.withColumn("order_date",col("order_date").cast("date"))\
    .withColumn("_ingest_timestamp",col("_ingest_timestamp").cast("timestamp"))

In [0]:
df_pharmacy_medication.limit(5).display()

## Write Pharmacy Medication into silver layer

In [0]:
df_pharmacy_medication.write.format("parquet")\
    .mode("overwrite")\
    .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/pharmacy_medication_order")

# READ BRONZE LAYER DATA - CLAIMS LINE

In [0]:
df_claims_claim_line=spark.read.format("parquet")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/raw_claims_claim_line.parquet")

In [0]:
df_claims_claim_line.limit(5).display()

In [0]:
df_claims_claim_line=df_claims_claim_line.filter(col("billed_amount")>=col("paid_amount"))

In [0]:
df_claims_claim_line=df_claims_claim_line.withColumn("service_date",col("service_date").cast("date"))\
    .withColumn("_ingest_timestamp",col("_ingest_timestamp").cast("timestamp"))

## Write the Claims Line into silver layer

In [0]:
df_claims_claim_line.write.format("parquet")\
    .mode("overwrite")\
        .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/claims_claim_line")

# READ THE BRONZE LAYER - PROVIDER

In [0]:
df_hr_provider=spark.read.format("parquet")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/raw_hr_provider.parquet")

In [0]:
df_hr_provider.display()

## NPI validation

In [0]:
df_hr_provider= df_hr_provider.filter(col("npi").rlike("^[0-9]{10}$"))

In [0]:
df_hr_provider.count()

## Duplication Check

In [0]:
df_hr_provider.groupBy("npi").count().filter("count>1").display()

## Write the HR Provider into silver layer

In [0]:
df_hr_provider.write.format("parquet")\
    .mode("overwrite")\
        .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/hr_provider")

# READ THE BRONZE LAYER - FACILITY

In [0]:
df_facility=spark.read.format("parquet")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/raw_ref_facility.parquet")

In [0]:
df_facility.limit(5).display()

## Write the Facility into Silver Layer

In [0]:
df_facility.write.format("parquet")\
    .mode("overwrite")\
        .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/facility")

# READ THE BRONZE LAYER - PAYER

In [0]:
df_payer=spark.read.format("parquet")\
    .option("inferSchema", "true")\
        .load("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/raw_ref_payer.parquet")

In [0]:
df_payer.limit(5).display()

## Write the Payer data into silver layer

In [0]:
df_payer.write.format("parquet")\
    .mode("overwrite")\
        .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/payer")

# READ THE BRONZE LAYER - icd10_diagnosis

In [0]:
df_icd10_diagnosis=spark.read.json("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/ref_icd10_diagnosis.json")
df_icd10_diagnosis.limit(5).display()

## Write the icd10 into silver layer

In [0]:
df_icd10_diagnosis.write.format("parquet")\
    .mode("overwrite")\
        .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/icd10_diagnosis")

# READ THE BRONZE LAYER - CPT PROCEDURE,DATE,IONIC LABTEST, NDC MEDICATION

In [0]:
df_cpt_procedure=spark.read.json("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/ref_cpt_procedure.json")
df_date=spark.read.json("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/ref_dim_date.json")
df_ionic_labtest=spark.read.json("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/ref_loinc_labtest.json")
df_ndc_medication=spark.read.json("abfss://bronze@healthcaredatalakedeb.dfs.core.windows.net/rawdata/ref_ndc_medication.json")


In [0]:
df_cpt_procedure.limit(5).display()

In [0]:
df_date.limit(5).display()

In [0]:
df_ionic_labtest.limit(5).display()

In [0]:
df_ndc_medication.limit(5).display()

## Write the CPT PROCEDURE,DATE,IONIC LABTEST, NDC MEDICATION into silver layer

In [0]:
df_cpt_procedure.write.format("parquet")\
    .mode("overwrite")\
        .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/cpt_procedure")
df_date.write.format("parquet")\
    .mode("overwrite")\
        .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/date")
df_ionic_labtest.write.format("parquet")\
    .mode("overwrite")\
        .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/ionic_labtest")
df_ndc_medication.write.format("parquet")\
    .mode("overwrite")\
        .save("abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/ndc_medication")